# W04 · Building the PEPS wrapper / 建立 PEPS wrapper

**English.** We assemble the full pipeline `Project -> Encode -> Aggregate ->
Model` (paper Eq. 8) and confirm that identity-encoder PEPS features are
affinely equivalent to APE features. We then swap in a learned grid to get
**Grid-PEPS**.

**繁體中文.** 組裝完整流程 `投影->編碼->聚合->模型`(論文式 8),確認 identity
編碼器的 PEPS 特徵與 APE 特徵仿射等價;再換入可學習 grid 得到 **Grid-PEPS**。

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import torch, matplotlib.pyplot as plt
from peps.train import auto_device
device = auto_device(); print('device', device)

device cuda


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## 1. Assemble PEPS by hand / 手動組裝 PEPS

In [2]:
from peps import Projector, GridEncoder, MLP, PEPS, make_aggregator
L = 6
proj = Projector(num_frequencies=L)
enc  = GridEncoder(dim=2, resolution=128, feature_dim=4)
agg  = make_aggregator('concat', proj.num_points, enc.feature_dim)
mlp  = MLP(agg.out_dim, out_dim=3, hidden_dim=64, num_layers=4)
model = PEPS(proj, enc, agg, mlp)
print('points', proj.num_points, '| agg out', agg.out_dim,
      '| params', sum(p.numel() for p in model.parameters()))
print(model(torch.rand(8, 2)).shape)

points 13 | agg out 52 | params 77443
torch.Size([8, 3])


## 2. Identity PEPS is affinely equivalent to APE / Identity PEPS 與 APE 仿射等價

In [3]:
from peps import AbsolutePositionalEncoding
x = torch.rand(400, 2)
peps_feat = proj(x).reshape(x.shape[0], -1)
ape_feat  = AbsolutePositionalEncoding(2, L, include_input=True)(x)
A = torch.cat([ape_feat, torch.ones(x.shape[0], 1)], 1)
resid = (A @ torch.linalg.lstsq(A, peps_feat).solution - peps_feat).abs().max()
print(f'affine residual APE->PEPS(identity): {resid.item():.2e}')

affine residual APE->PEPS(identity): 7.75e-07


## 3. Eq. (8) input-delta ablation / 式(8) 輸入 delta 消融
The full Eq. (8) is `M(A(E(P_1)..E(P_{2L+1})), delta)`. Setting `delta=True`
concatenates the raw coords to the aggregated vector before the decoder — a
few extra params that let the MLP see exact position, not only sampled latents.
With a **learned grid** the latents already encode position, so we expect the
effect to be small/noisy; we average over seeds and report honestly. It is off
by default.

式(8) 完整為 `M(A(E(P_1)..E(P_{2L+1})), delta)`。`delta=True` 把原始座標接到聚合
向量後再進解碼器——只多幾個參數,讓 MLP 看到精確位置而非僅取樣的 latent。由於
**可學習 grid** 的 latent 已編碼位置,預期效果小且有雜訊;故對多個 seed 取平均、
誠實呈現。預設關閉。

In [4]:
from apps.image.data import load_image, image_to_coords_targets, find_kodak
from apps.image.build import build_grid_peps
from peps.train import fit, TrainConfig, render_full
from peps.metrics import psnr
import numpy as np
img = load_image(find_kodak(1), max_size=256)
coords, targets, (H, W) = image_to_coords_targets(img)
def run_delta(delta, seed, steps=1500):
    torch.manual_seed(seed)
    model, pc = build_grid_peps(resolution=96, feature_dim=6, num_frequencies=6, aggregator='concat', delta=delta)
    fit(model, coords, targets, TrainConfig(steps=steps, batch_size=32768, lr=1e-2, device=device))
    pred = render_full(model, coords, device=device).reshape(H, W, 3).clamp(0, 1)
    return pc, psnr(pred, img)
# average over seeds: the grid_sample backward is non-deterministic on GPU,
# and with a learned grid the delta effect is small, so a single run is noisy.
seeds = [0, 1, 2]
off = [run_delta(False, s) for s in seeds]
on  = [run_delta(True,  s) for s in seeds]
off_pc, on_pc = off[0][0], on[0][0]
off_ps, on_ps = float(np.mean([p for _, p in off])), float(np.mean([p for _, p in on]))
for i, s in enumerate(seeds):
    print(f'seed {s}: delta_off={off[i][1]:.2f}  delta_on={on[i][1]:.2f}  ({on[i][1]-off[i][1]:+.2f} dB)')
print(f'mean/{len(seeds)} seeds: off={off_ps:.2f}  on={on_ps:.2f}  effect={on_ps-off_ps:+.2f} dB  (+{on_pc-off_pc} params)')
import csv, os
os.makedirs('../results', exist_ok=True)
with open('../results/delta_ablation.csv', 'w', newline='') as f:
    w = csv.writer(f, lineterminator='\n'); w.writerow(['delta', 'params', 'psnr_mean', 'n_seeds'])
    w.writerow(['off', off_pc, round(off_ps, 3), len(seeds)])
    w.writerow(['on', on_pc, round(on_ps, 3), len(seeds)])
print('saved ../results/delta_ablation.csv')

seed 0: delta_off=41.77  delta_on=41.56  (-0.21 dB)
seed 1: delta_off=41.63  delta_on=39.89  (-1.74 dB)
seed 2: delta_off=39.67  delta_on=42.71  (+3.04 dB)
mean/3 seeds: off=41.02  on=41.38  effect=+0.36 dB  (+128 params)
saved ../results/delta_ablation.csv


## 4. Takeaway / 小結
The wrapper is the reusable object every application uses, and `delta` is the
optional Eq. (8) skip. Next week we train Grid-PEPS on Kodak and reproduce
Table 1. 下週在 Kodak 上訓 Grid-PEPS 重現 Table 1。